# 01 — Build the team-season dataset

Turns the audited inputs into one modelling panel: one row per team-season, the §2.1 target, the
§2.3 features, and the market line where it exists.

**Three rules this notebook exists to enforce:**

1. **Training is not limited to line-covered seasons.** Every leakage-safe season before test year
   *T* trains — the 2015 fold trains on 2002–2014, not on 2014 alone. 2023 has no line and still
   trains for the 2024/2025 folds.
2. **Covers rows define evaluation rows and B0 only** — never who trains.
3. **Tier C is locked mechanically**, not by comment: prices never enter the panel, and the
   recursive guard in `tier_lock.py` blocks side / EV / confidence vocabulary from being written or
   exported — at any nesting depth.

**Amendment 2 (2026-08-03)** adds the venue correction — the old `home_games` counted nominal
home designations including neutral and international sites — and declares **M5**, a market-anchored
residual track, before any model exists. M1–M4 stay independent and structural.

**Reads:** `artifacts/data_audit.json` (gate + frozen folds), `data/schedules_snapshot.parquet`,
`data/pbp_team_season_epa.parquet` (pinned), `data/win_totals.csv`.
**Writes:** `data/team_season_panel.parquet`, `data/season_schedule_context.parquet`,
`artifacts/dataset_metadata.json`, `artifacts/preseason_feature_feasibility.json`.

```bash
papermill futures/season_team_totals/01_build_dataset.ipynb /tmp/out.ipynb
```

## Section 1 — Parameters

Everything resolves from the audit artifact rather than being restated here: fold sets, complete
seasons, hashes and the tier all come from `data_audit.json`. Only paths and switches live here.

In [ ]:
AUDIT_PATH      = None      # None -> futures/artifacts/data_audit.json
PANEL_PATH      = None      # None -> futures/data/team_season_panel.parquet
META_PATH       = None      # None -> futures/artifacts/dataset_metadata.json
LINES_PATH      = None      # None -> the file the audit actually audited
PBP_SNAPSHOT    = None      # None -> futures/data/pbp_team_season_epa.parquet
REFRESH_PBP     = False     # True = rebuild the PBP aggregate from nflreadpy and re-pin it
OFFLINE         = None      # None -> $APP_OFFLINE; True forbids network (snapshot required)
WRITE_ARTIFACTS = True
SEED            = 20260802
RUN_TESTS       = True

### Interpreting the output

Prints nothing — papermill replaces the cell. Nothing here can change *which* seasons train or
evaluate; that is read from the artifact in Section 2.

### What these tests guard

That no fold, season or tier can be injected as a parameter. A notebook that could be told its own
evaluation seasons would make the frozen fold set advisory.

In [ ]:
if RUN_TESTS:
    assert isinstance(SEED, int)
    for _n in ("FOLDS", "TEST_SEASONS", "TRAIN_SEASONS", "TIER", "TIER_C_OPEN"):
        assert _n not in dir(), f"{_n} must come from data_audit.json, never from a parameter"
    print(f"✓ Section 1 tests passed | paths+seed only; folds and tier come from the audit artifact")

### Reading the test result

Confirms the parameter surface is paths and a seed. Does **not** prove the artifact exists — that is
Section 2.

## Section 2 — Gate on the audit

Loads `data_audit.json` and refuses to run on `NO-GO`. Reads the frozen headline folds, the A1.4
strict-subset folds, the complete-season list, the input hashes, the feature-availability
classification, and `tier_c_open` — which is propagated into every artifact this notebook writes.

In [ ]:
import hashlib
import json
import platform
import re
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def _find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "app.py").exists() and (p / "futures").is_dir():
            return p
    raise RuntimeError(f"repo root not found above {start}")


REPO     = _find_repo_root(Path.cwd())
FUTURES  = REPO / "futures"
DATA_DIR = FUTURES / "data"
ART_DIR  = FUTURES / "artifacts"

AUDIT = Path(AUDIT_PATH) if AUDIT_PATH else ART_DIR / "data_audit.json"
PANEL = Path(PANEL_PATH) if PANEL_PATH else DATA_DIR / "team_season_panel.parquet"
META  = Path(META_PATH) if META_PATH else ART_DIR / "dataset_metadata.json"
AUDIT, PANEL, META = (p if p.is_absolute() else REPO / p for p in (AUDIT, PANEL, META))


def _rel(p) -> str:
    p = Path(p)
    try:
        return p.resolve().relative_to(REPO).as_posix()
    except ValueError:
        return str(p.resolve())


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_frame(df: pd.DataFrame) -> str:
    return hashlib.sha256(
        pd.util.hash_pandas_object(df.reset_index(drop=True), index=False).values.tobytes()
    ).hexdigest()


if not AUDIT.exists():
    raise RuntimeError(f"no audit artifact at {AUDIT} — run 00_data_audit.ipynb first")
audit = json.loads(AUDIT.read_text(encoding="utf-8"))

VERDICT = audit["verdict"]
if not VERDICT.startswith("GO"):
    raise RuntimeError(f"audit verdict is {VERDICT} — §5 stops the subproject here; 01 must not run")

TIER            = audit.get("tier_available", "unknown")
TIER_C_OPEN     = bool(audit.get("tier_c_open", False))
FOLDS           = list(audit["folds"]["test_seasons"])
FOLDS_STRICT    = list(audit["folds_strict_sensitivity"]["test_seasons"])
COMPLETE_SEASONS = [int(s) for s in audit["outcomes"]["complete_seasons"]]
PREDICT_SEASON  = int(audit["predict_season"]["season"])
TARGET_COL      = audit["target"]["column"]
AVAILABLE_FAMILIES = sorted(f["feature_family"] for f in audit["feature_availability"]
                            if f["verdict"] == "AVAILABLE")
UNAVAILABLE_FAMILIES = sorted(f["feature_family"] for f in audit["feature_availability"]
                              if f["verdict"] != "AVAILABLE")
AUDIT_LINES_FILE = audit["lines"]["file"]
AUDIT_LINES_HASH = audit["lines"]["file_sha256"]
AUDIT_SCHED_HASH = audit["schedule"]["hash"]
AUDIT_OUTCOME_HASH = audit["outcomes"]["hash"]

RUN_AT = datetime.now(timezone.utc)
PROVENANCE = {"notebook": "futures/season_team_totals/01_build_dataset.ipynb",
              "run_at_utc": RUN_AT.isoformat(), "as_of_date": RUN_AT.date().isoformat(),
              "python": sys.version.split()[0], "platform": platform.platform(),
              "pandas": pd.__version__, "numpy": np.__version__, "seed": SEED,
              "audit_verdict": VERDICT, "audit_artifact": _rel(AUDIT)}

print(f"verdict          : {VERDICT}   (tier {TIER}, gate C open: {TIER_C_OPEN})")
print(f"headline folds   : {FOLDS}")
print(f"A1.4 strict folds: {FOLDS_STRICT}")
print(f"complete seasons : {COMPLETE_SEASONS[0]}–{COMPLETE_SEASONS[-1]} ({len(COMPLETE_SEASONS)})")
print(f"target           : {TARGET_COL}")
print(f"AVAILABLE families ({len(AVAILABLE_FAMILIES)}): {AVAILABLE_FAMILIES}")

### Interpreting the output

`GO-TIER-B`, gate C shut, **10 headline folds** and **4 strict folds**, 24 complete seasons, target
`wins_half_ties`, 7 AVAILABLE families. Those are read, not decided — this notebook has no way to
widen them.

### What these tests guard

The gate is real: a non-GO verdict raises before any data is read. Folds must be non-empty, ordered,
inside the complete-season list, and the strict set must be a subset of the headline set.

In [ ]:
if RUN_TESTS:
    assert VERDICT in ("GO", "GO-TIER-B")
    assert FOLDS and FOLDS == sorted(FOLDS), "headline folds missing or unordered"
    assert set(FOLDS_STRICT) <= set(FOLDS), "A1.4 subset is not a subset"
    assert all(s in COMPLETE_SEASONS for s in FOLDS), "a fold has no settled outcome"
    assert PREDICT_SEASON not in COMPLETE_SEASONS, "the predict season must have no settled outcome"
    assert TARGET_COL == "wins_half_ties"
    assert TIER_C_OPEN is False or VERDICT == "GO", "tier_c_open may only be True on a full GO"
    assert "season_S_results" in UNAVAILABLE_FAMILIES and "season_S_pbp" in UNAVAILABLE_FAMILIES
    print(f"✓ Section 2 tests passed | {VERDICT}, {len(FOLDS)} folds, {len(FOLDS_STRICT)} strict, "
          f"tier_c_open={TIER_C_OPEN}, target={TARGET_COL}")

### Reading the test result

The gate held and the fold sets are internally consistent. Does **not** prove the inputs on disk are
the ones the audit blessed — Section 3 checks that by hash.

## Section 3 — Load and hash-verify the inputs

Loads the pinned schedule snapshot and the market file, then checks both against the hashes recorded
by the audit. A panel built on different bytes than the audit blessed is not an audited panel.

In [ ]:
SNAP  = DATA_DIR / "schedules_snapshot.parquet"
LINES = Path(LINES_PATH) if LINES_PATH else (REPO / AUDIT_LINES_FILE)
LINES = LINES if LINES.is_absolute() else REPO / LINES

FRANCHISE_MAP = {"OAK": "LV", "SD": "LAC", "STL": "LA"}

# The audit hashes its OWN window (SEASON_MIN..predict season), so filter identically before
# hashing — otherwise a wider frame produces a different hash for the same underlying data.
SEASON_MIN_AUDIT = int(audit["outcomes"]["season_min"])
SEASON_MAX_AUDIT = max(int(audit["outcomes"]["season_max"]), PREDICT_SEASON)

sched = pd.read_parquet(SNAP)
sched = sched[(sched["game_type"] == "REG") &
              sched["season"].between(SEASON_MIN_AUDIT, SEASON_MAX_AUDIT)].copy()
sched["gameday"] = pd.to_datetime(sched["gameday"])
for _s in ("home", "away"):
    sched[f"{_s}_franchise"] = sched[f"{_s}_team"].replace(FRANCHISE_MAP)
SCHED_HASH = sha256_frame(sched[["game_id", "season", "week", "home_team", "away_team", "result"]])

lines = pd.read_csv(LINES)
LINES_HASH = sha256_file(LINES)
lines["franchise"] = lines["team"].astype(str).str.upper().replace(FRANCHISE_MAP)

print(f"schedule : {_rel(SNAP)}  {len(sched):,} REG games  hash {SCHED_HASH[:16]}…")
print(f"           audit hash {AUDIT_SCHED_HASH[:16]}…  match={SCHED_HASH == AUDIT_SCHED_HASH}")
print(f"lines    : {_rel(LINES)}  {len(lines):,} rows  hash {LINES_HASH[:16]}…")
print(f"           audit hash {str(AUDIT_LINES_HASH)[:16]}…  match={LINES_HASH == AUDIT_LINES_HASH}")
print(f"line seasons: {sorted(lines['season'].unique())}")

### Interpreting the output

Both hashes must match the audit's. The line file covers **11 seasons** — note what is *missing*
(2012, 2013, 2023) and remember those seasons still train; they simply never evaluate.

### What these tests guard

Byte-level agreement with the audited inputs, and that the market file carries no column this
notebook is forbidden to use — prices are Tier-C material and never enter the panel (Section 9).

In [ ]:
if RUN_TESTS:
    assert SCHED_HASH == AUDIT_SCHED_HASH, "schedule snapshot differs from the audited bytes"
    assert LINES_HASH == AUDIT_LINES_HASH, "line file differs from the audited bytes"
    assert {"season", "team", "win_total_line", "as_of_date"} <= set(lines.columns)
    assert lines["book"].isna().all(), "a named book appeared — re-run the audit; the tier may change"
    assert not lines.duplicated(["season", "franchise"]).any()
    print(f"✓ Section 3 tests passed | both inputs hash-match the audit; "
          f"{len(lines):,} market rows over {lines['season'].nunique()} seasons, book null throughout")

### Reading the test result

The inputs are the audited ones. Does **not** prove the market numbers are accurate — nothing in
this repo can check that.

## Section 4 — Outcome table

Rebuilds the team-season outcome table with the audit's own logic and asserts the hash matches, so
the target this panel carries is provably the target the audit verified.

In [ ]:
def build_outcomes(sched_src: pd.DataFrame) -> pd.DataFrame:
    played = sched_src[sched_src["result"].notna()]
    home = pd.DataFrame({"season": played["season"], "franchise": played["home_franchise"],
                         "margin": played["result"], "pf": played["home_score"],
                         "pa": played["away_score"]})
    away = pd.DataFrame({"season": played["season"], "franchise": played["away_franchise"],
                         "margin": -played["result"], "pf": played["away_score"],
                         "pa": played["home_score"]})
    tg = pd.concat([home, away], ignore_index=True)
    tg["win"] = (tg["margin"] > 0).astype(float)
    tg["loss"] = (tg["margin"] < 0).astype(float)
    tg["tie"] = (tg["margin"] == 0).astype(float)
    out = (tg.groupby(["season", "franchise"])
           .agg(games_played=("win", "size"), wins=("win", "sum"), losses=("loss", "sum"),
                ties=("tie", "sum"), points_for=("pf", "sum"), points_against=("pa", "sum"))
           .reset_index())
    out["wins_half_ties"] = out["wins"] + 0.5 * out["ties"]
    out["point_diff"] = out["points_for"] - out["points_against"]
    return out


outcomes = build_outcomes(sched)
outcomes = outcomes[outcomes["season"].isin(COMPLETE_SEASONS)].reset_index(drop=True)
OUTCOME_HASH = sha256_frame(outcomes[["season", "franchise", "games_played", "wins_half_ties"]])

print(f"outcome rows : {len(outcomes):,}  seasons {outcomes['season'].min()}–{outcomes['season'].max()}")
print(f"hash         : {OUTCOME_HASH[:16]}…   audit {AUDIT_OUTCOME_HASH[:16]}…  "
      f"match={OUTCOME_HASH == AUDIT_OUTCOME_HASH}")
print(outcomes.tail(3).to_string(index=False))

### Interpreting the output

**768 team-seasons, hash matching the audit.** If this ever disagrees, the panel and the audit are
describing different data and nothing downstream is trustworthy.

### What these tests guard

Hash identity with the audit, plus the conservation identity re-checked here rather than assumed
from the previous notebook — `Σ wins_half_ties == games played`, per season.

In [ ]:
if RUN_TESTS:
    assert OUTCOME_HASH == AUDIT_OUTCOME_HASH, "outcome table differs from the audited one"
    for _s in COMPLETE_SEASONS:
        _o = outcomes[outcomes["season"] == _s]
        _g = int(sched[(sched["season"] == _s) & sched["result"].notna()].shape[0])
        assert len(_o) == 32 and abs(float(_o["wins_half_ties"].sum()) - _g) < 1e-9
    assert outcomes[TARGET_COL].between(0, outcomes["games_played"]).all()
    print(f"✓ Section 4 tests passed | {len(outcomes):,} rows, hash matches the audit, "
          f"wins conserved in all {len(COMPLETE_SEASONS)} seasons")

### Reading the test result

Same table, independently rebuilt, same hash. Does **not** re-prove the schedule is correct — only
that this notebook derived the target identically.

## Section 5 — Prior-season PBP EPA (pinned snapshot)

`prior_season_pbp_epa` was preregistered **AVAILABLE**, so it is built now — before any model is
fitted. Adding it after seeing `02`'s results would be a feature choice informed by results, which
needs an amendment.

Raw play-by-play is far too large to pin, so what is pinned is the **derived team-season
aggregate**: offensive EPA/play, defensive EPA/play allowed, and offensive success rate, over REG
pass+run plays with non-null EPA. Fetched once, written with a provenance sidecar (nflreadpy
version, as-of date, sha256, seasons), and read from disk every run after.

Only **settled** seasons are fetched, and features read *S−1* — so the predict season is covered by
2025 and no season-S play ever enters.

In [ ]:
import os

PBP_SNAP = Path(PBP_SNAPSHOT) if PBP_SNAPSHOT else DATA_DIR / "pbp_team_season_epa.parquet"
PBP_SNAP = PBP_SNAP if PBP_SNAP.is_absolute() else REPO / PBP_SNAP
PBP_PROV = PBP_SNAP.with_suffix(".provenance.json")
if OFFLINE is None:
    OFFLINE = os.environ.get("APP_OFFLINE", "") == "1"
OFFLINE = bool(OFFLINE)


def build_pbp_aggregate(seasons):
    # Team-season EPA aggregate over REG pass+run plays with a computed EPA.
    import nflreadpy as nfl
    d = nfl.load_pbp(list(seasons)).to_pandas()
    d = d[(d["season_type"] == "REG") & d["play_type"].isin(["pass", "run"]) & d["epa"].notna()]
    off = (d.groupby(["season", "posteam"])
           .agg(off_plays=("epa", "size"), off_epa_play=("epa", "mean"),
                off_success_rate=("success", "mean")).reset_index()
           .rename(columns={"posteam": "team"}))
    dfn = (d.groupby(["season", "defteam"])
           .agg(def_plays=("epa", "size"), def_epa_play=("epa", "mean")).reset_index()
           .rename(columns={"defteam": "team"}))
    out = off.merge(dfn, on=["season", "team"], how="inner")
    out["season"] = out["season"].astype(int)
    return out.sort_values(["season", "team"]).reset_index(drop=True)


if PBP_SNAP.exists() and not REFRESH_PBP:
    epa_raw = pd.read_parquet(PBP_SNAP)
    PBP_SOURCE = f"snapshot:{_rel(PBP_SNAP)}"
    PBP_LIB = json.loads(PBP_PROV.read_text(encoding="utf-8")).get("nflreadpy") if PBP_PROV.exists() else None
elif OFFLINE:
    raise RuntimeError(f"OFFLINE and no PBP snapshot at {PBP_SNAP} — run once online to pin it")
else:
    import nflreadpy as _nfl
    epa_raw = build_pbp_aggregate(COMPLETE_SEASONS)
    PBP_LIB = getattr(_nfl, "__version__", "unknown")
    PBP_SOURCE = f"nflreadpy=={PBP_LIB} live pull"
    if WRITE_ARTIFACTS:
        epa_raw.to_parquet(PBP_SNAP, index=False)
        PBP_PROV.write_text(json.dumps(
            {"artifact": _rel(PBP_SNAP), "nflreadpy": PBP_LIB,
             "as_of_date": RUN_AT.date().isoformat(), "seasons": COMPLETE_SEASONS,
             "rows": int(len(epa_raw)), "sha256": sha256_file(PBP_SNAP),
             "definition": "REG pass+run plays with non-null EPA; off/def EPA per play and "
                           "offensive success rate, per team-season",
             "built_by": "futures/season_team_totals/01_build_dataset.ipynb"},
            indent=2), encoding="utf-8")

epa = epa_raw.copy()
epa["franchise"] = epa["team"].replace(FRANCHISE_MAP)
EPA_COLS = ["off_epa_play", "def_epa_play", "off_success_rate"]
PBP_HASH = sha256_file(PBP_SNAP) if PBP_SNAP.exists() else None

print(f"source   : {PBP_SOURCE}")
print(f"rows     : {len(epa):,}  seasons {epa['season'].min()}–{epa['season'].max()}  "
      f"sha256 {str(PBP_HASH)[:16]}…")
print(epa.groupby("season").agg(teams=("franchise", "nunique"), min_off_plays=("off_plays", "min"))
      .tail(3).to_string())
print(epa[["season", "franchise"] + EPA_COLS].tail(3).to_string(index=False))

### Interpreting the output

**768 team-seasons, 2002–2025**, 32 teams every season and never fewer than ~850 offensive plays —
so no season is thin enough to make a team mean unstable.

The snapshot is the pinned input: after the first run this is read from disk, and `REFRESH_PBP=True`
is the only way to change it. That keeps the feature reproducible without carrying gigabytes of raw
play-by-play in the repo.

### What these tests guard

Coverage matches the outcome table exactly (same seasons, 32 teams each), EPA values sit in a
plausible per-play range, and the snapshot **contains no season beyond the settled window** — a
predict-season row here would be same-season information one join away from the feature matrix.

In [ ]:
if RUN_TESTS:
    assert sorted(epa["season"].unique()) == sorted(COMPLETE_SEASONS), \
        "PBP snapshot seasons differ from the audited complete seasons"
    assert PREDICT_SEASON not in set(epa["season"]), \
        "the PBP snapshot must not contain the predict season"
    for _s, _g in epa.groupby("season"):
        assert _g["franchise"].nunique() == 32, f"{_s}: {_g['franchise'].nunique()} teams in the PBP aggregate"
    assert epa["off_epa_play"].between(-0.5, 0.5).all() and epa["def_epa_play"].between(-0.5, 0.5).all()
    assert epa["off_success_rate"].between(0.2, 0.8).all()
    assert epa["off_plays"].min() > 500, "a team-season with implausibly few plays"
    assert not epa.duplicated(["season", "franchise"]).any()
    print(f"✓ Section 5 tests passed | {len(epa):,} team-seasons of pinned EPA, "
          f"{epa['season'].min()}–{epa['season'].max()}, predict season absent")

### Reading the test result

The EPA source covers exactly the settled seasons and stops before the predict season. Does **not**
prove the features built from it are lagged correctly — Section 8's blinding probe does that, and it
now blinds this source too.

## Section 6 — Venue context (Amendment 2, A2.1)

The old `home_games` counted **nominal home designations**, including neutral and international
sites, so it never measured home-field exposure. A2.1 replaces it with four features and a
game-level venue authority.

**Venue key** = `INTL::<venue>` when the stadium name is in the pinned international table, else
`stadium_id`. `stadium_id` alone cannot work: in this snapshot it is the *nominal home team's* id —
`JAX00` spans seven stadium names including Wembley and Tottenham, and the 2026
Jacksonville–Philadelphia game at Tottenham is labelled `location = "Home"` with `stadium_id =
JAX00`. Using `stadium_id` for domestic venues absorbs renames (`SEA00` = CenturyLink → Lumen).

**Primary home venue** = modal venue key over designated-home games, excluding explicit `Neutral`.
A surviving tie **aborts** — never resolved by row order.

**Effective neutral** = `location == "Neutral"` **or** venue key ≠ the nominal home team's primary
venue key. Neutral ≠ international in either direction, and both count for **both** teams.

In [ ]:
# Pinned international venues, keyed by stadium NAME (see A2.1.1 for why not stadium_id).
# Canonical name -> the name variants this snapshot uses for that physical venue.
INTERNATIONAL_VENUES = {
    "wembley":            ["Wembley Stadium"],
    "tottenham":          ["Tottenham Stadium", "Tottenham Hotspur Stadium"],
    "twickenham":         ["Twickenham Stadium"],
    "azteca":             ["Azteca Stadium", "Estadio Banorte"],
    "munich":             ["Allianz Arena", "FC Bayern Munich Stadium"],
    "frankfurt":          ["Deutsche Bank Park"],
    "sao_paulo":          ["Arena Corinthians"],
    "rio":                ["Maracana Stadium"],
    "melbourne":          ["Melbourne Cricket Ground"],
    "paris":              ["Stade de France"],
    "madrid":             ["Bernabeu"],
    "toronto":            ["Rogers Centre"],
}
INTL_NAME_TO_VENUE = {n: v for v, names in INTERNATIONAL_VENUES.items() for n in names}


def build_venue_context(sched_src: pd.DataFrame) -> pd.DataFrame:
    # Game-level venue authority. Derived ONLY from the published schedule (teams, location,
    # stadium) — never from results, so blinding season T cannot change it.
    ctx = sched_src[["game_id", "season", "week", "home_franchise", "away_franchise",
                     "location", "stadium_id", "stadium"]].copy()
    ctx["international_game"] = ctx["stadium"].map(lambda s: str(s) in INTL_NAME_TO_VENUE)
    ctx["venue_key"] = [f"INTL::{INTL_NAME_TO_VENUE[str(s)]}" if str(s) in INTL_NAME_TO_VENUE
                        else str(sid) for s, sid in zip(ctx["stadium"], ctx["stadium_id"])]

    # primary home venue = modal venue key over designated-home games, excluding explicit Neutral
    dh = ctx[ctx["location"] != "Neutral"]
    cnt = dh.groupby(["season", "home_franchise", "venue_key"]).size().rename("n").reset_index()
    top = cnt[cnt["n"] == cnt.groupby(["season", "home_franchise"])["n"].transform("max")]
    ties = top.groupby(["season", "home_franchise"]).size()
    if (ties > 1).any():
        raise RuntimeError(f"unresolved modal venue tie (A2.1.2 aborts): "
                           f"{ties[ties > 1].index.tolist()}")
    prim = top.set_index(["season", "home_franchise"])["venue_key"]
    prim_sid = (dh.groupby(["season", "home_franchise", "venue_key"])["stadium_id"].first())

    ctx["primary_home_venue_key"] = [prim.get((s, h)) for s, h in
                                     zip(ctx["season"], ctx["home_franchise"])]
    ctx["primary_home_stadium_id"] = [prim_sid.get((s, h, prim.get((s, h))))
                                      for s, h in zip(ctx["season"], ctx["home_franchise"])]
    ctx["effective_neutral"] = ((ctx["location"] == "Neutral") |
                                (ctx["venue_key"] != ctx["primary_home_venue_key"]))
    return ctx.sort_values(["season", "week", "game_id"], kind="mergesort").reset_index(drop=True)


def venue_team_season(ctx: pd.DataFrame) -> pd.DataFrame:
    # Team-season venue aggregates. Neutral/international games count for BOTH teams (A2.1.3).
    home = ctx.rename(columns={"home_franchise": "franchise"})[
        ["season", "franchise", "effective_neutral", "international_game"]].copy()
    home["designated_home"] = 1.0
    home["true_home_venue"] = (~ctx["effective_neutral"]).astype(float).values
    away = ctx.rename(columns={"away_franchise": "franchise"})[
        ["season", "franchise", "effective_neutral", "international_game"]].copy()
    away["designated_home"] = 0.0
    away["true_home_venue"] = 0.0
    both = pd.concat([home, away], ignore_index=True)
    return (both.groupby(["season", "franchise"])
            .agg(designated_home_games=("designated_home", "sum"),
                 true_home_venue_games=("true_home_venue", "sum"),
                 neutral_site_games=("effective_neutral", "sum"),
                 international_games=("international_game", "sum"))
            .reset_index())


venue_ctx = build_venue_context(sched)
venue_ts = venue_team_season(venue_ctx)

_recl = venue_ctx[(venue_ctx["location"] != "Neutral") & venue_ctx["effective_neutral"]]
venue_by_season = (venue_ctx.groupby("season")
                   .agg(games=("game_id", "size"),
                        explicit_neutral=("location", lambda s: int((s == "Neutral").sum())),
                        effective_neutral=("effective_neutral", "sum"),
                        international=("international_game", "sum")).reset_index())
venue_by_season["reclassified_from_home"] = [
    int(((_recl["season"] == s)).sum()) for s in venue_by_season["season"]]

print(f"venue context: {len(venue_ctx):,} games, {venue_ctx['venue_key'].nunique()} venue keys")
print(f"international venues pinned: {len(INTERNATIONAL_VENUES)} "
      f"({sum(len(v) for v in INTERNATIONAL_VENUES.values())} name variants)")
print()
print(venue_by_season[venue_by_season[["explicit_neutral", "effective_neutral",
                                       "international", "reclassified_from_home"]].sum(axis=1) > 0]
      .to_string(index=False))
print()
print("--- designated-HOME games reclassified as effectively neutral ---")
print(_recl[["season", "week", "home_franchise", "away_franchise", "stadium",
             "venue_key", "primary_home_venue_key"]].to_string(index=False))

### Interpreting the output

The reclassification table is the correction made visible. It catches two distinct things:

* **Domestic relocations** the snapshot marks `Home` — New Orleans 2005 (Giants Stadium, Alamo
  Dome ×3), Buffalo's Toronto series 2009–2012, Minnesota 2010 at TCF Bank.
* **The 2026 Jacksonville–Philadelphia game at Tottenham**, labelled `location = "Home"` with
  Jacksonville's own `stadium_id`. An id-only or `Neutral`-only rule misses it entirely.

Buffalo's Rogers Centre games are both *effectively neutral* and *international* while being marked
`Home` — which is why the two flags are computed separately rather than one implying the other.

### What these tests guard

The A2.1 rules, plus the five 2026 checks Amendment 2 requires by name. The venue build is also
asserted to be **independent of results** — recomputed on a fully results-blinded schedule it must
return an identical frame, since it may read only teams, location and stadium.

In [ ]:
if RUN_TESTS:
    # --- structural ---
    assert len(venue_ctx) == len(sched) and not venue_ctx["game_id"].duplicated().any()
    assert venue_ctx["primary_home_venue_key"].notna().all(), "a team-season has no primary venue"
    assert len(venue_ts) == 32 * sched["season"].nunique()   # PANEL_SEASONS is defined later
    # every game contributes to exactly 2 teams
    assert float(venue_ts["designated_home_games"].sum()) == float(len(venue_ctx))
    assert float(venue_ts["neutral_site_games"].sum()) == 2.0 * float(venue_ctx["effective_neutral"].sum())
    assert float(venue_ts["international_games"].sum()) == 2.0 * float(venue_ctx["international_game"].sum())
    assert (venue_ts["true_home_venue_games"] <= venue_ts["designated_home_games"]).all()

    # --- the five 2026 checks required by Amendment 2 ---
    def _chk(gid):
        r = venue_ctx[venue_ctx["game_id"] == gid]
        assert len(r) == 1, f"{gid} not found"
        return r.iloc[0]

    for _gid, _label in (("2026_04_IND_WAS", "Washington-Indianapolis, Tottenham"),
                         ("2026_05_PHI_JAX", "Jacksonville-Philadelphia, Tottenham"),
                         ("2026_06_HOU_JAX", "Jacksonville-Houston, Wembley"),
                         ("2026_10_NE_DET", "Detroit-New England, Munich")):
        _r = _chk(_gid)
        assert bool(_r["effective_neutral"]), f"{_label}: not effective-neutral"
        assert bool(_r["international_game"]), f"{_label}: not international"
    # the one labelled Home must still be caught
    assert _chk("2026_05_PHI_JAX")["location"] == "Home", "fixture changed; the Home-labelled case is the point"
    # Jacksonville's London games must not inflate true home-venue games
    _jax26 = venue_ts[(venue_ts["season"] == 2026) & (venue_ts["franchise"] == "JAX")].iloc[0]
    assert _jax26["international_games"] >= 2, "JAX 2026 should carry both London games"
    assert _jax26["true_home_venue_games"] == _jax26["designated_home_games"] - 2, \
        "JAX London games must not count as true home venue games"
    # a normal home stadium game still counts
    _norm = venue_ctx[(venue_ctx["season"] == 2025) & (~venue_ctx["effective_neutral"])]
    assert len(_norm) > 200 and not _norm["international_game"].any()

    # --- venue depends only on the published schedule ---
    _blind = sched.copy()
    _blind[["result", "home_score", "away_score", "total"]] = np.nan
    assert build_venue_context(_blind).equals(venue_ctx), \
        "venue context changed when results were blinded — it is reading an outcome"
    print(f"✓ Section 6 tests passed | {len(venue_ctx):,} games, "
          f"{int(venue_ctx['effective_neutral'].sum())} effectively neutral "
          f"({len(_recl)} reclassified from designated-home), "
          f"{int(venue_ctx['international_game'].sum())} international; 2026 checks pass")

### Reading the test result

The four named 2026 games are effective-neutral **and** international, the Home-labelled Tottenham
game among them, and Jacksonville's two London trips do not count as true home venue games. The
blinding equality proves the venue build reads no outcome.

Does **not** estimate any venue *effect* — it corrects an exposure count. No Jacksonville-London
familiarity term is introduced (A2.1.4).

## Section 7 — Features (AVAILABLE families only)

One builder over the schedule frame and the pinned EPA snapshot, so Section 9 can re-run it
against **blinded** copies of both and prove nothing depends on season *S*.

* From *S−1* and earlier: record, points for/against, Pythagorean expectation, decayed 3-year form,
  coach career win%, **and off/def EPA per play + success rate**.
* From *S*'s **published schedule** (no results): games scheduled, the four A2.1 venue counts,
  division games, rest, bye week, opponent strength measured on *S−1*, and a coach-change flag.

All seven AVAILABLE families are now built — nothing is deferred, so no feature decision remains
open once `02` starts producing numbers.

In [ ]:
DEFERRED_FAMILIES = {}          # every AVAILABLE family is built; nothing left open
PYTHAG_EXP = 2.37


def build_features(sched_src: pd.DataFrame, epa_src: pd.DataFrame, seasons: list[int]) -> pd.DataFrame:
    # Team-season features. Every column is derived from seasons < S, or from S's
    # published schedule (opponents/dates/coaches), never from S's results.
    hist = build_outcomes(sched_src)                      # settled seasons only
    hist = hist.set_index(["season", "franchise"])
    epa_idx = epa_src.set_index(["season", "franchise"])
    # venue is rebuilt from the same (possibly blinded) schedule, so the blinding probe covers it
    ven_idx = venue_team_season(build_venue_context(sched_src)).set_index(["season", "franchise"])

    # week-1 coach per team-season, from the published schedule (no results needed)
    wk1 = sched_src[sched_src["week"] == sched_src.groupby("season")["week"].transform("min")]
    coach = pd.concat([
        pd.DataFrame({"season": wk1["season"], "franchise": wk1["home_franchise"], "coach": wk1["home_coach"]}),
        pd.DataFrame({"season": wk1["season"], "franchise": wk1["away_franchise"], "coach": wk1["away_coach"]}),
    ], ignore_index=True).drop_duplicates(["season", "franchise"]).set_index(["season", "franchise"])["coach"]

    # Coach career record strictly through S-1, from settled games only.
    # Pivot -> reindex over the FULL season range -> cumsum -> shift(1), so the value at season S
    # is defined even when S has no played games (the predict season). Indexing by played-season
    # rows instead would leave every predict-season row NaN — a train-present/deploy-absent gap.
    played = sched_src[sched_src["result"].notna()]
    cg = pd.concat([
        pd.DataFrame({"season": played["season"], "coach": played["home_coach"],
                      "w": (played["result"] > 0).astype(float) + 0.5 * (played["result"] == 0)}),
        pd.DataFrame({"season": played["season"], "coach": played["away_coach"],
                      "w": (played["result"] < 0).astype(float) + 0.5 * (played["result"] == 0)}),
    ], ignore_index=True)
    cg = cg.groupby(["coach", "season"]).agg(w=("w", "sum"), g=("w", "size")).reset_index()
    _cols = list(range(int(sched_src["season"].min()), int(sched_src["season"].max()) + 2))
    _cw = (cg.pivot(index="coach", columns="season", values="w")
             .reindex(columns=_cols).fillna(0.0).cumsum(axis=1).shift(1, axis=1))
    _cg = (cg.pivot(index="coach", columns="season", values="g")
             .reindex(columns=_cols).fillna(0.0).cumsum(axis=1).shift(1, axis=1))

    # season-S schedule structure (published before Week 1)
    sh = sched_src.rename(columns={"home_franchise": "franchise", "away_franchise": "opp",
                                   "home_rest": "rest"})[["season", "week", "franchise", "opp", "rest", "div_game"]]
    sh["is_home"] = 1.0
    sa = sched_src.rename(columns={"away_franchise": "franchise", "home_franchise": "opp",
                                   "away_rest": "rest"})[["season", "week", "franchise", "opp", "rest", "div_game"]]
    sa["is_home"] = 0.0
    tg = pd.concat([sh, sa], ignore_index=True)

    rows = []
    for season in seasons:
        sg = tg[tg["season"] == season]
        for franchise, g in sg.groupby("franchise"):
            r = {"season": int(season), "franchise": franchise}
            # --- schedule structure of season S (opponents are preseason information) ---
            r["games_scheduled"] = int(len(g))
            v = ven_idx.loc[(season, franchise)]
            r["designated_home_games"] = float(v["designated_home_games"])
            r["true_home_venue_games"] = float(v["true_home_venue_games"])
            r["neutral_site_games"] = float(v["neutral_site_games"])
            r["international_games"] = float(v["international_games"])
            r["div_games"] = float(g["div_game"].sum())
            r["mean_rest"] = float(g["rest"].mean())
            r["has_bye"] = float(len(set(range(int(g["week"].min()), int(g["week"].max()) + 1)) - set(g["week"])) > 0)
            # --- prior-season outcome features (S-1) ---
            k1 = (season - 1, franchise)
            if k1 in hist.index:
                p = hist.loc[k1]
                r["prior_games"] = float(p["games_played"])
                r["prior_wins"] = float(p["wins_half_ties"])
                r["prior_win_pct"] = float(p["wins_half_ties"] / p["games_played"])
                r["prior_point_diff"] = float(p["point_diff"])
                r["prior_points_for"] = float(p["points_for"])
                r["prior_points_against"] = float(p["points_against"])
                _pf, _pa = float(p["points_for"]), float(p["points_against"])
                r["prior_pythag_win_pct"] = _pf ** PYTHAG_EXP / (_pf ** PYTHAG_EXP + _pa ** PYTHAG_EXP)
            else:
                for c in ("prior_games", "prior_wins", "prior_win_pct", "prior_point_diff",
                          "prior_points_for", "prior_points_against", "prior_pythag_win_pct"):
                    r[c] = np.nan
            # --- prior-season EPA (S-1), from the pinned snapshot ---
            if k1 in epa_idx.index:
                e = epa_idx.loc[k1]
                r["prior_off_epa_play"] = float(e["off_epa_play"])
                r["prior_def_epa_play"] = float(e["def_epa_play"])
                r["prior_epa_diff"] = float(e["off_epa_play"]) - float(e["def_epa_play"])
                r["prior_off_success_rate"] = float(e["off_success_rate"])
            else:
                for c in ("prior_off_epa_play", "prior_def_epa_play", "prior_epa_diff",
                          "prior_off_success_rate"):
                    r[c] = np.nan
            # --- decayed 3-season form (S-1, S-2, S-3) ---
            num = den = 0.0
            for lag, wgt in ((1, 3.0), (2, 2.0), (3, 1.0)):
                k = (season - lag, franchise)
                if k in hist.index:
                    p = hist.loc[k]
                    num += wgt * float(p["wins_half_ties"] / p["games_played"])
                    den += wgt
            r["form_3yr_win_pct"] = num / den if den else np.nan
            # --- opponent strength: S's opponents, measured on S-1 ---
            ow = [float(hist.loc[(season - 1, o), "wins_half_ties"] / hist.loc[(season - 1, o), "games_played"])
                  for o in g["opp"] if (season - 1, o) in hist.index]
            r["sos_prior_win_pct"] = float(np.mean(ow)) if ow else np.nan
            # --- coach features ---
            c_now = coach.get((season, franchise), None)
            c_prev = coach.get((season - 1, franchise), None)
            r["coach_changed"] = float(c_now != c_prev) if (c_now is not None and c_prev is not None) else np.nan
            if c_now is not None and c_now in _cg.index and season in _cg.columns:
                cwv, cgv = float(_cw.loc[c_now, season]), float(_cg.loc[c_now, season])
            else:
                cwv, cgv = 0.0, 0.0          # first-time coach: no prior games, not a missing value
            # NaN only at the left edge of the window (the first season has no history in scope) —
            # left-censored, deliberately not collapsed to a false zero.
            r["coach_prior_games"] = cgv
            r["coach_prior_win_pct"] = float(cwv / cgv) if (cgv == cgv and cgv >= 16) else np.nan
            rows.append(r)
    # Column order is the contract — enforced here, never left to dict insertion order.
    return (pd.DataFrame(rows)[["season", "franchise"] + FEATURE_COLS]
            .sort_values(["season", "franchise"]).reset_index(drop=True))


FEATURE_COLS = ["games_scheduled",
                "designated_home_games", "true_home_venue_games",
                "neutral_site_games", "international_games",
                "div_games", "mean_rest", "has_bye",
                "prior_games", "prior_wins", "prior_win_pct", "prior_point_diff",
                "prior_points_for", "prior_points_against", "prior_pythag_win_pct",
                "prior_off_epa_play", "prior_def_epa_play", "prior_epa_diff",
                "prior_off_success_rate",
                "form_3yr_win_pct", "sos_prior_win_pct",
                "coach_changed", "coach_prior_win_pct", "coach_prior_games"]

# the four features the pinned PBP snapshot contributes (success rate has no "epa" in its name,
# so it must be named explicitly rather than matched by substring)
PBP_FEATURE_COLS = ["prior_off_epa_play", "prior_def_epa_play", "prior_epa_diff",
                    "prior_off_success_rate"]

PANEL_SEASONS = sorted(set(COMPLETE_SEASONS) | {PREDICT_SEASON})
features = build_features(sched, epa, PANEL_SEASONS)

print(f"feature rows: {len(features):,}   columns: {len(FEATURE_COLS)}")
print(features[["season", "franchise"] + FEATURE_COLS].isna().sum().to_frame("nulls").T.to_string())
print(features[features["season"] == PREDICT_SEASON].head(2).to_string(index=False))

### Interpreting the output

**800 rows** (25 seasons × 32) and **24 features** — 21 before Amendment 2, with `home_games`
replaced by the four A2.1 venue counts. Nulls concentrate in 2002 — the first season in
scope has no prior, so every `prior_*` column including EPA is null there — and
`coach_prior_win_pct` is null under 16 career games, a stated threshold rather than an imputation.
The predict season builds cleanly: schedule structure present, prior-season features from 2025.

### What these tests guard

Shape and grain (32 teams every season, no duplicate team-seasons), the pinned feature list matching
the frame, and — the substantive one — **no feature may be null for the predict season that is
populated in training**. That is the deploy-gap check this repo added after a train-present /
deploy-absent feature collapsed a projection model.

In [ ]:
if RUN_TESTS:
    assert len(features) == 32 * len(PANEL_SEASONS)
    assert not features.duplicated(["season", "franchise"]).any()
    assert list(features.columns) == ["season", "franchise"] + FEATURE_COLS, "feature order drifted"
    # deploy-gap check: predict-season coverage must not collapse vs the training seasons
    _tr = features[features["season"].isin(COMPLETE_SEASONS[1:])]
    _pr = features[features["season"] == PREDICT_SEASON]
    _gaps = {c: (round(100 * _tr[c].notna().mean(), 1), round(100 * _pr[c].notna().mean(), 1))
             for c in FEATURE_COLS
             if _tr[c].notna().mean() - _pr[c].notna().mean() > 0.20}
    assert not _gaps, f"deploy gap >20pp (train%, predict%): {_gaps}"
    # every built feature belongs to an AVAILABLE family; deferred ones are recorded, not silent
    assert not DEFERRED_FAMILIES, f"an AVAILABLE family is still deferred: {DEFERRED_FAMILIES}"
    assert set(PBP_FEATURE_COLS) <= set(FEATURE_COLS), "the preregistered PBP/EPA family is not in the matrix"
    assert len(PBP_FEATURE_COLS) == 4
    print(f"✓ Section 7 tests passed | {len(features):,} rows × {len(FEATURE_COLS)} features "
          f"({len(PBP_FEATURE_COLS)} from the pinned PBP family), no deploy gap >20pp, "
          f"nothing deferred")

### Reading the test result

No feature is populated in training and absent for 2026. Does **not** prove the features are
*point-in-time clean* — that is Section 7, and it is a different kind of check.

## Section 8 — Assemble the panel and join the market

Left-joins the target onto features (predict season keeps a null target) and left-joins the market
line, flagging `line_covered`. **The join is left, deliberately:** a season with no line keeps its
rows so it can train.

In [ ]:
panel = features.merge(
    outcomes[["season", "franchise", "games_played", "wins", "losses", "ties",
              TARGET_COL, "points_for", "points_against", "point_diff"]],
    on=["season", "franchise"], how="left")

_mkt = lines[["season", "franchise", "win_total_line", "as_of_date", "market_source",
              "point_in_time_status"]].copy()
_mkt = _mkt.rename(columns={"win_total_line": "market_line", "as_of_date": "market_as_of",
                            "point_in_time_status": "market_pit_status"})
panel = panel.merge(_mkt, on=["season", "franchise"], how="left")

panel["line_covered"] = panel["market_line"].notna()
panel["is_predict_season"] = panel["season"] == PREDICT_SEASON
panel["has_target"] = panel[TARGET_COL].notna()
panel["market_strictly_dated"] = panel["market_pit_status"].eq("strictly_before_week1")
panel = panel.sort_values(["season", "franchise"], kind="mergesort").reset_index(drop=True)

_by = (panel.groupby("season")
       .agg(rows=("franchise", "size"), with_target=("has_target", "sum"),
            line_covered=("line_covered", "sum"), strict=("market_strictly_dated", "sum"))
       .reset_index())
print(f"panel: {len(panel):,} rows × {panel.shape[1]} cols")
print(_by.to_string(index=False))

### Interpreting the output

**800 rows.** Read the three right-hand columns together: 768 rows carry a target, **352** carry a
market line, **160** of those are strictly dated. The seasons showing `with_target=32,
line_covered=0` — 2002–2013 and 2023 — are exactly the rows that **train but never evaluate**. That
gap is the point of the left join.

### What these tests guard

That the join did not drop or duplicate rows, that market coverage matches the audit exactly
(352 rows over 11 seasons), and that the predict season carries features with **no** target and
**no** line.

In [ ]:
if RUN_TESTS:
    assert len(panel) == len(features), "the market join changed the row count"
    assert not panel.duplicated(["season", "franchise"]).any()
    assert int(panel["line_covered"].sum()) == len(lines) == audit["lines"]["rows_valid"], \
        "market coverage disagrees with the audited file"
    assert sorted(panel.loc[panel["line_covered"], "season"].unique()) == audit["lines"]["usable_seasons"]
    _p = panel[panel["is_predict_season"]]
    assert len(_p) == 32 and not _p["has_target"].any() and not _p["line_covered"].any()
    # seasons that train but never evaluate must actually exist — the whole point of the left join
    _train_only = sorted(set(panel.loc[panel["has_target"], "season"]) -
                         set(panel.loc[panel["line_covered"], "season"]))
    assert 2023 in _train_only, "2023 must be present with a target and no line"
    print(f"✓ Section 8 tests passed | {len(panel):,} rows, {int(panel['line_covered'].sum())} line-covered "
          f"over {panel.loc[panel['line_covered'], 'season'].nunique()} seasons; "
          f"{len(_train_only)} train-only seasons incl. 2023")

### Reading the test result

13 train-only seasons, 2023 among them. Does **not** yet prove they are *used* in training — Section
9 defines eligibility and checks it.

## Section 9 — Leakage tests

The decisive one is a **blinding probe**: for each fold season *T*, null every season-*T* result
**and** every season-*T* EPA value, rebuild the features from scratch, and require the season-*T*
rows back **bit-identical**. If any feature could see *T*, blinding must change something.

Also: a column-name ban on outcome fields, and a check that no feature is a transform of the target.

In [ ]:
blind_report = []
for T in FOLDS + [PREDICT_SEASON]:
    blinded = sched.copy()
    blinded.loc[blinded["season"] == T, ["result", "home_score", "away_score", "total"]] = np.nan
    blinded_epa = epa.copy()                       # blind the PBP source for season T as well
    blinded_epa.loc[blinded_epa["season"] == T, EPA_COLS] = np.nan
    reb = build_features(blinded, blinded_epa, [T])
    orig = features[features["season"] == T].reset_index(drop=True)
    reb = reb[orig.columns].reset_index(drop=True)
    same = orig.equals(reb) or np.allclose(orig[FEATURE_COLS].astype(float),
                                           reb[FEATURE_COLS].astype(float), equal_nan=True)
    diffs = [c for c in FEATURE_COLS
             if not np.allclose(orig[c].astype(float), reb[c].astype(float), equal_nan=True)]
    blind_report.append({"season": int(T), "identical": bool(same), "changed_features": diffs})

TARGET_DERIVED = {TARGET_COL, "wins", "losses", "ties", "games_played", "points_for",
                  "points_against", "point_diff"}
_leaky_names = [c for c in FEATURE_COLS
                if c in TARGET_DERIVED or re.search(r"(?<!prior_)(^|_)(actual|result|outcome)(_|$)", c)]

print(f"{'season':<9}{'identical':<12}changed features")
for r in blind_report:
    print(f"{r['season']:<9}{str(r['identical']):<12}{r['changed_features'] or '—'}")
print(f"\nfeature names colliding with target-derived columns: {_leaky_names or 'none'}")
print(f"target and target-derived columns kept OUT of FEATURE_COLS: "
      f"{sorted(TARGET_DERIVED - set(FEATURE_COLS)) == sorted(TARGET_DERIVED)}")

### Interpreting the output

Every fold and the predict season come back **identical** under blinding. Erasing season *T*'s
scores, margins, totals **and EPA** changes not one feature value for *T*, so no feature can be
reading them — directly or through a chain.

`prior_*` columns are unaffected by design: they read *S−1*, which blinding leaves alone.

### What these tests guard

The probe must be **able to fail** — so it is inverted on two deliberately leaky features, one per
source: current-season point differential (schedule) and current-season offensive EPA (snapshot).
Both must be detected. A blinding test that passes on a known-leaky column proves nothing.

In [ ]:
if RUN_TESTS:
    _bad = [r for r in blind_report if not r["identical"]]
    assert not _bad, f"features change when season outcomes are blinded — leak: {_bad}"
    assert not _leaky_names, f"target-derived names in the feature list: {_leaky_names}"
    assert not (TARGET_DERIVED & set(FEATURE_COLS)), "a target-derived column is in the feature matrix"

    # RED CONTROL: inject a same-season feature and require the probe to catch it
    def _leaky_builder(src, esrc, seasons):
        f = build_features(src, esrc, seasons)
        cur = build_outcomes(src).set_index(["season", "franchise"])
        ecur = esrc.set_index(["season", "franchise"])
        f["LEAK_current_point_diff"] = [
            float(cur.loc[(s, t), "point_diff"]) if (s, t) in cur.index else np.nan
            for s, t in zip(f["season"], f["franchise"])]
        f["LEAK_current_off_epa"] = [
            float(ecur.loc[(s, t), "off_epa_play"]) if (s, t) in ecur.index else np.nan
            for s, t in zip(f["season"], f["franchise"])]
        return f

    _T = FOLDS[0]
    _be = epa.copy()
    _be.loc[_be["season"] == _T, EPA_COLS] = np.nan
    _b = sched.copy()
    _b.loc[_b["season"] == _T, ["result", "home_score", "away_score", "total"]] = np.nan
    _o = _leaky_builder(sched, epa, [_T])
    _r = _leaky_builder(_b, _be, [_T])
    for _lk in ("LEAK_current_point_diff", "LEAK_current_off_epa"):
        assert not np.allclose(_o[_lk].astype(float), _r[_lk].astype(float), equal_nan=True), \
            f"the blinding probe FAILED TO DETECT the injected leak {_lk} — the probe is not a test"
    print(f"✓ Section 9 tests passed | {len(blind_report)} seasons blinded (schedule + EPA), "
          f"all identical; both red controls caught their injected same-season leak")

### Reading the test result

The probe caught both injected leaks and cleared every real fold — so the clean result is evidence,
not an artifact of a test that cannot fail. Does **not** cover leakage introduced *later*, in `02`'s
preprocessing or selection; those are that notebook's assertions.

## Section 10 — Training and evaluation eligibility

The rule Joseph set: **train on every leakage-safe season before *T*, whether or not it has a line;
evaluate only on line-covered rows in *T*.** Encoded as one function so `02`–`05` cannot re-derive it
differently, and materialised as an explicit table.

In [ ]:
def fold_rows(panel_df: pd.DataFrame, T: int) -> dict:
    # Training rows: every settled team-season strictly before T (line or no line).
    # Evaluation rows: line-covered rows in T. Market rows never gate training.
    tr = panel_df[(panel_df["season"] < T) & panel_df["has_target"]]
    ev = panel_df[(panel_df["season"] == T) & panel_df["line_covered"] & panel_df["has_target"]]
    return {"test_season": int(T),
            "train_seasons": sorted(int(s) for s in tr["season"].unique()),
            "n_train": int(len(tr)), "n_eval": int(len(ev)),
            "train_seasons_without_line": sorted(
                int(s) for s in set(tr["season"]) - set(panel_df.loc[panel_df["line_covered"], "season"]))}


eligibility = [fold_rows(panel, T) for T in FOLDS]
eligibility_strict = [fold_rows(panel, T) for T in FOLDS_STRICT]

print(f"{'fold':<7}{'train seasons':<16}{'n_train':<9}{'n_eval':<8}train seasons with NO line")
for e in eligibility:
    span = f"{e['train_seasons'][0]}–{e['train_seasons'][-1]}"
    print(f"{e['test_season']:<7}{span:<16}{e['n_train']:<9}{e['n_eval']:<8}{e['train_seasons_without_line']}")
print(f"\nA1.4 strict folds: {[e['test_season'] for e in eligibility_strict]} "
      f"(eval rows {[e['n_eval'] for e in eligibility_strict]})")

### Interpreting the output

The 2015 fold trains on **2002–2014, 416 rows** — not on 2014 alone, which is what restricting
training to line-covered seasons would have produced. Every fold's `train seasons with NO line` column
lists 2002–2013, and from 2024 onward it includes **2023**: exactly the behaviour Joseph specified.

Evaluation is 32 rows per fold, all line-covered. Training grows monotonically; evaluation never does.

### What these tests guard

Three things, each a way this could go wrong silently: no training row is from season ≥ *T*; training
is **strictly larger** than the line-covered subset (proving the market did not gate training); and
the two named cases are asserted by hand — 2015 trains on 13 seasons, and 2023 trains for 2024/2025.

In [ ]:
if RUN_TESTS:
    _line_seasons = set(panel.loc[panel["line_covered"], "season"])
    for e in eligibility:
        T = e["test_season"]
        assert max(e["train_seasons"]) < T, f"fold {T} trains on season >= T"
        assert e["n_eval"] > 0 and e["n_train"] > 0
        # the market must not have gated training
        _line_only = len(panel[(panel["season"] < T) & panel["line_covered"] & panel["has_target"]])
        assert e["n_train"] > _line_only, \
            f"fold {T}: training ({e['n_train']}) is not larger than the line-covered subset ({_line_only})"
        assert e["train_seasons_without_line"], f"fold {T} trains on no line-free season"
    _f15 = next(e for e in eligibility if e["test_season"] == 2015)
    assert _f15["train_seasons"] == list(range(2002, 2015)) and _f15["n_train"] == 32 * 13, \
        f"2015 must train on 2002–2014 (416 rows), got {_f15['n_train']} over {_f15['train_seasons']}"
    for _T in (2024, 2025):
        _e = next(e for e in eligibility if e["test_season"] == _T)
        assert 2023 in _e["train_seasons"] and 2023 in _e["train_seasons_without_line"], \
            f"2023 must train for the {_T} fold despite having no market line"
    assert all(e["n_eval"] == 32 for e in eligibility)
    print(f"✓ Section 10 tests passed | 2015 trains on 2002–2014 ({_f15['n_train']} rows); "
          f"2023 trains for 2024 and 2025 without a line; every fold trains beyond the market subset")

### Reading the test result

Training is decoupled from market coverage, verified on the two cases that would have exposed the
error. Does **not** prove `02` will *use* this function — which is why it ships in the panel metadata
and `02` asserts against it.

## Section 11 — Preseason feature feasibility (Amendment 2, A2.4)

A frozen verdict for each proposed QB / All-Pro / injury / roster family: can it be built
point-in-time, honestly, for both history and 2026? **No family here is added to `FEATURE_COLS`** —
A2.4 forbids CONDITIONAL and UNAVAILABLE, and states that AVAILABLE families are not added by this
amendment either.

High importance in the spread model is **not** evidence for this project. Prose version, with the
full reasoning and the exact unlock for each blocked family: `futures/PRESEASON_FEATURE_NOTES.md`.

In [ ]:
# verdicts frozen 2026-08-03, before notebook 02 existed. "burden"/"PBP" wording keeps the
# artifact clear of the Tier-C vocabulary the guard scans for.
FEASIBILITY = [
    ("expected_preseason_starting_qb", "schedule home_qb_id/away_qb_id", 1999, False, False, False,
     "UNAVAILABLE", "populated after games are completed; 272/272 null for 2026 in the pinned "
                    "snapshot and 0 null for settled seasons"),
    ("starting_qb_change", "derived from expected_preseason_starting_qb", 1999, False, False, False,
     "UNAVAILABLE", "derived from the expected starting QB, whose source field is populated only "
                    "after games are completed; inherits that blocker exactly"),
    ("prior_qb_passing_epa_per_snap", "PBP prior season + starter identity", 1999, True, False, False,
     "CONDITIONAL", "prior-season PBP is settled, but attaching it to the 2026 starter needs a "
                    "dated preseason depth snapshot this repo does not own"),
    ("prior_passer_rating", "nextgen stats (2016+) + starter identity", 2016, True, False, False,
     "CONDITIONAL", "same identity blocker; source also starts 2016, leaving 2002-2015 uncovered"),
    ("prior_cpoe", "nextgen stats (2016+) + starter identity", 2016, True, False, False,
     "CONDITIONAL", "same identity blocker; same 2016 coverage floor"),
    ("current_roster_weighted_allpro", "nfl_allpro_1997_2025.csv + a dated preseason roster", 1997,
     False, False, False, "UNAVAILABLE",
     "the All-Pro file has no player id and its Team column is the honoring team, not the current "
     "one; mapping honored players to current teams needs a dated preseason roster"),
    ("current_roster_allpro_off_def_split", "same as current_roster_weighted_allpro", 1997,
     False, False, False, "UNAVAILABLE",
     "same missing identity spine and same absent dated preseason roster as the weighted variant"),
    ("roster_continuity_returning_snaps", "roster + snap tables", 2012, False, False, False,
     "CONDITIONAL", "roster tables are revised in place; no dated preseason snapshot is owned"),
    ("preseason_ir_pup_nfi_status", "weekly in-season injury report", 2009, False, False, False,
     "UNAVAILABLE", "the injury feed is a weekly in-season report; no preseason status snapshot "
                    "exists, and a week-1 report is published in game week"),
    ("expected_games_lost_injury_burden", "derived from preseason_ir_pup_nfi_status", 2009,
     False, False, False, "UNAVAILABLE",
     "derived from preseason status, for which no dated preseason snapshot exists; inherits that "
     "blocker exactly"),
    ("offensive_line_continuity", "depth charts + roster", 2012, False, False, False,
     "UNAVAILABLE", "depth-chart coverage ends at 2024 in this stack, so it is absent for the "
                    "2026 deploy season"),
    ("prior_season_turnover_rate", "PBP prior season", 1999, True, True, True,
     "AVAILABLE", "settled before the target season, no identity join; NOT added - the pinned PBP "
                  "aggregate carries no turnover column, and A2.4 adds no family"),
    ("prior_season_special_teams_epa", "PBP prior season", 1999, True, True, True,
     "AVAILABLE", "settled before the target season, no identity join; NOT added - same reason"),
]
_F_COLS = ["family", "source", "earliest_season", "source_is_point_in_time",
           "historical_preseason_snapshot_exists", "buildable_for_2026", "verdict", "reason"]
feasibility = pd.DataFrame(FEASIBILITY, columns=_F_COLS)
feasibility["included_in_feature_cols"] = False

FEAS_PATH = ART_DIR / "preseason_feature_feasibility.json"
feasibility_doc = {
    "written_by": "futures/season_team_totals/01_build_dataset.ipynb",
    "authority": "PREREGISTRATION.md §10 Amendment 2 (A2.4)",
    "frozen_before": "any implementation or execution of notebook 02",
    "rule": "CONDITIONAL and UNAVAILABLE families may not enter FEATURE_COLS; AVAILABLE families "
            "are not added by Amendment 2 either",
    "spread_model_importance_is_not_evidence": {
        "why": "its top feature is ~the closing line (corr 0.994), and the ranking's history "
               "records a closing-line interpretation problem, a sack-history leak and an "
               "All-Pro identity collision",
        "corrected_published_result": "HIGH 129/238 = 54.2017%, Wilson lower 47.8551%, below the "
                                      "52.4% break-even; no selection band clears it",
        "reference": "betting/experiments/audit_2026-08-03c_final/PROVENANCE.md",
    },
    "counts": {v: int((feasibility["verdict"] == v).sum())
               for v in ("AVAILABLE", "CONDITIONAL", "UNAVAILABLE")},
    "families": feasibility.to_dict(orient="records"),
    "prose": "futures/PRESEASON_FEATURE_NOTES.md",
}

print(feasibility[["family", "verdict", "buildable_for_2026"]].to_string(index=False))
print()
print(feasibility_doc["counts"])

### Interpreting the output

**2 AVAILABLE, 4 CONDITIONAL, 7 UNAVAILABLE — and 0 added to the feature list.**

Three blockers explain nearly all of it: the schedule's QB fields are post-game (272/272 null for
2026); the All-Pro file has no player id and names the *honoring* team, so "current-roster talent"
needs a dated preseason roster nobody owns; and injury status is a weekly in-season feed, not a
preseason snapshot.

The two AVAILABLE families are genuinely buildable but stay out: adding either means re-pinning the
PBP aggregate, and A2.4 adds no family.

### What these tests guard

That the verdict vocabulary is closed, that no family in this table leaked into `FEATURE_COLS`, and
that the artifact round-trips. The QB blocker is also **re-measured live** rather than asserted from
prose — the notebook counts the null QB fields in the predict season itself.

In [ ]:
if RUN_TESTS:
    assert set(feasibility["verdict"]) <= {"AVAILABLE", "CONDITIONAL", "UNAVAILABLE"}
    assert not feasibility["family"].duplicated().any() and len(feasibility) == 13
    assert not feasibility["included_in_feature_cols"].any(), "A2.4: no audited family may be added"
    for _f in feasibility["family"]:
        assert _f not in FEATURE_COLS, f"{_f} leaked into FEATURE_COLS"
    # CONDITIONAL/UNAVAILABLE must not be buildable-for-2026 without a named blocker
    _bad = feasibility[(feasibility["verdict"] != "AVAILABLE") & feasibility["buildable_for_2026"]]
    assert _bad.empty, f"non-AVAILABLE family marked buildable: {_bad['family'].tolist()}"
    assert (feasibility["reason"].str.len() > 30).all(), "every verdict needs a stated reason"
    # re-measure the QB blocker on the pinned snapshot instead of trusting the table
    _qb = sched[sched["season"] == PREDICT_SEASON]
    assert _qb["home_qb_id"].isna().all() and len(_qb) == 272, \
        "the QB blocker was asserted but not reproduced on the snapshot"
    assert sched[sched["season"] == 2025]["home_qb_id"].notna().all()
    if WRITE_ARTIFACTS:
        FEAS_PATH.write_text(json.dumps(feasibility_doc, indent=2, default=str), encoding="utf-8")
        _back = json.loads(FEAS_PATH.read_text(encoding="utf-8"))
        assert len(_back["families"]) == 13 and _back["counts"]["UNAVAILABLE"] == 7
    print(f"✓ Section 11 tests passed | 13 families verdicted "
          f"({feasibility_doc['counts']}), none added to FEATURE_COLS; "
          f"QB blocker reproduced live ({len(_qb)}/272 null for {PREDICT_SEASON})")

### Reading the test result

The QB blocker is reproduced from the snapshot in this run — 272/272 null for 2026 against zero
nulls for 2025 — so the verdict rests on a measurement, not on the note that describes it.

Does **not** claim any of these families would help. Feasibility is a precondition, not evidence.

## Section 12 — Tier-C lock (mechanical)

`tier_c_open=false` has to be enforceable, not advisory. Three mechanisms:

1. **Prices never enter the panel.** `price_over` / `price_under` exist in the market file and are
   deliberately not joined — the panel physically cannot compute EV or break-even.
2. **`tier_lock.assert_no_tier_c`** — an importable module, not code exec-ed out of this notebook —
   walks DataFrames, mappings, sequences and bare strings **recursively**, so a banned token nested
   inside a dict value is caught, not just a key.
3. **The lock travels in the metadata**, so a downstream notebook reads its own restrictions.

In [ ]:
sys.path.insert(0, str(FUTURES / "season_team_totals"))
from tier_lock import (TIER_C_BANNED, TierCViolation, assert_no_tier_c, is_text_dtype,
                       tokens as _tokens)

# Exempt ONLY the exact proper name(s) the audited provenance uses. An exact-literal allowlist,
# never a token one: "Covers Sports Odds History" passes, "…History Plus" still fails.
# {VERDICT, TIER} are the audit's own verdict vocabulary ("GO-TIER-B", "A+B") — provenance that
# names the tier system, not a Tier-C claim.
# "play" is banned as wagering vocabulary ("best play of the week"), but it is also the football
# unit of EPA. These EXACT column names are exempt; "best play of the week" still fails.
FOOTBALL_LITERALS = {"prior_off_epa_play", "prior_def_epa_play", "off_epa_play", "def_epa_play",
                     "off_plays", "def_plays"}
# Reviewed repository paths that happen to contain banned vocabulary. Exact strings only — the
# bare token stays banned, so "a betting edge" still fails.
PATH_LITERALS = {"betting/experiments/audit_2026-08-03c_final/PROVENANCE.md"}
ALLOWED_LITERALS = (set(audit["lines"].get("market_sources", []))
                    | set(lines["market_source"].dropna().astype(str).unique())
                    | set(lines["point_in_time_status"].dropna().astype(str).unique())
                    | FOOTBALL_LITERALS
                    | PATH_LITERALS
                    | {VERDICT, TIER})


def guard(obj, where: str) -> None:
    assert_no_tier_c(obj, where, allowed_literals=ALLOWED_LITERALS, tier_c_open=TIER_C_OPEN)


PRICE_COLS_WITHHELD = [c for c in ("price_over", "price_under") if c in lines.columns]
guard(panel, "panel")

LOCK = {
    "tier_c_open": TIER_C_OPEN,
    "tier_available": TIER,
    "authority": "PREREGISTRATION §7 gate C + §10 Amendment 1 A1.5",
    "price_columns_withheld_from_panel": PRICE_COLS_WITHHELD,
    "why": "gate C requires a named book; the market source is an archived consensus with book=null, "
           "so no priced quantity may be computed or exported",
    "forbidden_downstream": ["side", "over/under recommendation", "probability vs a posted line",
                             "confidence tier", "expected value", "break-even", "profitability",
                             "bet / edge / lock / value / play language"],
    "permitted": ["projected wins", "win distribution", "accuracy vs the archived market consensus "
                  "(aggregate, reported for both the headline and the A1.4 strict subset)"],
    "market_naming": "archived market consensus of unattributed sportsbook origin — never "
                     "'the sportsbook line', 'Vegas', or 'the market'",
    "guard": "futures/season_team_totals/tier_lock.py :: assert_no_tier_c(obj, where, "
             "allowed_literals=..., tier_c_open=...) — import it; recursive over frames, mappings, "
             "sequences and strings. Call before writing any artifact.",
    "exact_closing_timestamps_available": False,
}

print(f"tier_c_open            : {TIER_C_OPEN}")
print(f"prices withheld        : {PRICE_COLS_WITHHELD} (present in the market file, not in the panel)")
print(f"panel columns          : {len(panel.columns)} — guard clean")
print(f"banned vocabulary      : {len(TIER_C_BANNED)} tokens")

### Interpreting the output

`price_over` / `price_under` exist in `win_totals.csv` and are **absent from the panel** — the lock is
structural before it is textual. The guard runs clean over the panel.

One narrow exemption: `*_epa_play` names use "play" as the football unit, so those **exact** names are
allowlisted. The token still bites everywhere else — `"best play of the week"` is a red control.

The guard also asserts its own dtype gate: pandas 2 types a string column `object`, pandas 3 types it
`str`, and an `== object` test would silently skip every value scan on pandas 3.

`tier_c_open` is copied from the audit, never computed here, so this notebook cannot promote itself.

### What these tests guard

That the guard **can fail**, including the nested case that defeated the first version:
`{"headline": "best bet of the week"}` passed when the guard checked mapping keys but never
descended into values. Red controls now cover nested dicts, lists, DataFrames, a near-miss literal
and a bare string; green controls (`games_played`, `market_line`, the audited source name) must
still pass, because a guard that rejects everything gets disabled by the first person it annoys.

In [ ]:
if RUN_TESTS:
    assert TIER_C_OPEN is False, "this build assumes a closed gate C"
    assert not (set(panel.columns) & {"price_over", "price_under"}), "prices leaked into the panel"
    # red controls — the guard must reject. The last two prove the market-source exemption is an
    # EXACT-literal allowlist: a near-miss and an unrelated odds phrase must both still fail.
    _reds = [{"headline": "best bet of the week"},                    # the nested false negative
             {"a": {"b": ["fine", "expected value here"]}},
             {"results": [{"note": "confidence HIGH"}]},
             ["ok", ("ok", {"k": "kelly stake"})],
             "best odds available",
             pd.DataFrame({"ev_estimate": [1.0]}),
             pd.DataFrame({"confidence": ["HIGH"]}),
             pd.DataFrame({"note": ["best bet of the week"]}),
             pd.DataFrame({"price_over": [-110]}),
             pd.DataFrame({"market_source": ["Covers Sports Odds History Plus"]}),
             {"note": "best play of the week"},          # 'play' still bites outside the exact names
             {"note": "a betting edge on this game"},    # 'betting' still bites outside the paths
             pd.DataFrame({"epa_play_recommendation": [1.0]})]
    _caught = 0
    for _bad in _reds:
        try:
            guard(_bad, "red-control")
        except TierCViolation:
            _caught += 1
    assert _caught == len(_reds), f"guard rejected only {_caught}/{len(_reds)} red controls"
    # green controls — the guard must accept
    guard(pd.DataFrame({"games_played": [17], "prior_win_pct": [0.5], "market_line": [8.5],
                        "market_source": ["Covers Sports Odds History"]}), "green-control")
    guard({"nested": {"deep": ["games_played", "strictly_before_week1", 3, None]}}, "green-nested")
    assert ALLOWED_LITERALS and all(isinstance(v, str) for v in ALLOWED_LITERALS)
    assert "Covers Sports Odds History" in ALLOWED_LITERALS
    assert FOOTBALL_LITERALS <= ALLOWED_LITERALS and "play" not in ALLOWED_LITERALS,         "the football exemption must be exact column names, never the bare token"
    assert PATH_LITERALS <= ALLOWED_LITERALS and "betting" not in ALLOWED_LITERALS,         "the path exemption must be exact paths, never the bare token"
    # cross-version: the value scan only runs on columns is_text_dtype() accepts. pandas 2 gives a
    # string column dtype `object`; pandas 3 gives `str`. An `== object` gate skips the whole scan
    # under pandas 3 and value-only violations escape — assert the detector works on THIS pandas.
    assert is_text_dtype(pd.Series(["a", "b"])),         f"is_text_dtype rejects text under pandas {pd.__version__} — value scan would be skipped"
    assert not is_text_dtype(pd.Series([1.0, 2.0]))
    guard(panel[FEATURE_COLS], "features")
    guard(feasibility_doc, "feasibility")        # the Section 11 artifact, guarded once guard() exists
    assert LOCK["tier_c_open"] is False and LOCK["exact_closing_timestamps_available"] is False
    print(f"✓ Section 12 tests passed | recursive guard rejected {_caught}/{len(_reds)} red controls "
          f"(nested dicts, lists, frames, near-miss + football-token literals), accepted both green controls; "
          f"prices withheld; lock closed; value scan verified live on pandas {pd.__version__}")

### Reading the test result

The guard discriminates at depth — it rejects the nested-dict case, lists, frames and a near-miss
literal, while accepting `games_played`, `market_line` and a nested clean structure. Does **not** stop someone from deleting the
call in a later notebook; it makes doing so a visible edit rather than an omission.

## Section 13 — Write the panel and its metadata

Deterministic parquet (sorted, pinned column order) plus `dataset_metadata.json`: feature list **in
order**, both fold sets with their eligibility tables, input hashes, the lock, deferred families, and
provenance.

In [ ]:
# M5 eligibility (A2.2): line-covered seasons strictly before T. Recorded, not fitted.
m5_training_by_fold = []
for _T in FOLDS:
    _tr = panel[(panel["season"] < _T) & panel["has_target"] & panel["line_covered"]]
    m5_training_by_fold.append({
        "test_season": int(_T),
        "train_seasons": sorted(int(s) for s in _tr["season"].unique()),
        "n_train": int(len(_tr)),
        "usable_inner_validation_seasons": max(0, _tr["season"].nunique() - 1),
        "takes_fallback_alpha": bool(_tr["season"].nunique() - 1 < 2),
    })

VENUE_PATH = DATA_DIR / "season_schedule_context.parquet"
VENUE_COLS = ["game_id", "season", "week", "home_franchise", "away_franchise", "location",
              "stadium_id", "stadium", "venue_key", "primary_home_venue_key",
              "primary_home_stadium_id", "effective_neutral", "international_game"]
venue_out = venue_ctx[VENUE_COLS].copy()
guard(venue_out, "venue_context")
if WRITE_ARTIFACTS:
    venue_out.to_parquet(VENUE_PATH, index=False)
VENUE_HASH = sha256_frame(venue_out)

PANEL_COLS = (["season", "franchise", "is_predict_season", "has_target", "line_covered",
               "market_strictly_dated"] + FEATURE_COLS +
              [TARGET_COL, "wins", "losses", "ties", "games_played",
               "points_for", "points_against", "point_diff",
               "market_line", "market_as_of", "market_source", "market_pit_status"])
panel_out = panel[PANEL_COLS].copy()
guard(panel_out, "panel_out")

if WRITE_ARTIFACTS:
    panel_out.to_parquet(PANEL, index=False)
PANEL_HASH = sha256_frame(panel_out)

metadata = {
    "notebook": "futures/season_team_totals/01_build_dataset.ipynb",
    "panel": {"path": _rel(PANEL), "rows": int(len(panel_out)), "columns": PANEL_COLS,
              "frame_sha256": PANEL_HASH,
              "file_sha256": (sha256_file(PANEL) if WRITE_ARTIFACTS and PANEL.exists() else None),
              "grain": "one row per (season, franchise)"},
    "target": {"column": TARGET_COL, "rows_with_target": int(panel_out["has_target"].sum()),
               "settlement_assumption": audit["target"]["settlement_assumption"]},
    "venue_contract": {
        "authority": "PREREGISTRATION.md §10 Amendment 2 (A2.1)",
        "retired_feature": "home_games (counted nominal home designations incl. neutral and "
                           "international sites; never measured home-field exposure)",
        "features": ["designated_home_games", "true_home_venue_games", "neutral_site_games",
                     "international_games"],
        "venue_key_rule": "INTL::<venue> when the stadium name is in the pinned international "
                          "table, else stadium_id",
        "why_not_stadium_id_alone": "stadium_id is the nominal home team's id in this snapshot: "
                                    "JAX00 spans 7 stadium names incl. Wembley and Tottenham, and "
                                    "the 2026 JAX-PHI game at Tottenham is labelled location=Home "
                                    "with stadium_id=JAX00",
        "why_stadium_id_for_domestic": "it absorbs renames a name key would split (SEA00 = "
                                       "CenturyLink Field -> Lumen Field, mid-2020)",
        "primary_home_venue_rule": "modal venue key over designated-home games, excluding explicit "
                                   "Neutral; a surviving tie ABORTS the build",
        "effective_neutral_rule": "location == Neutral OR venue key != nominal home team's primary "
                                  "venue key",
        "counts_for_both_teams": True,
        "no_team_specific_adjustment": "no Jacksonville-London familiarity term is introduced; it "
                                       "is not estimable from this sample",
        "international_venues": INTERNATIONAL_VENUES,
        "observed_name_id_pairs": sorted({(str(r.stadium), str(r.stadium_id))
                                          for r in venue_ctx[venue_ctx.international_game]
                                          .itertuples()}),
        "artifact": _rel(VENUE_PATH),
        "by_season": venue_by_season.to_dict(orient="records"),
    },
    "model_tracks": {
        "m1_m4_independent_structural": {
            "features": FEATURE_COLS,
            "market_line_in_features": False,
            "note": "independent structural projections; market_line must never be silently "
                    "inserted into this track",
        },
        "m5_market_anchored": {
            "features": FEATURE_COLS,
            "offset": "market_line",
            "note": "market-anchored residual projection; cannot be described as independently "
                    "beating a sportsbook or a market",
        },
    },
    "m5_contract": {
        "authority": "PREREGISTRATION.md §10 Amendment 2 (A2.2)",
        "definition": "residual = wins_half_ties - market_line; prediction = market_line + "
                      "predicted_residual",
        "market_line_role": "offset / base prediction, never a quote",
        "withheld_quote_columns": "recorded in the panel restriction manifest",
        "training_rows_require": ["target", "archived consensus line"],
        "training_window": "line-covered team-seasons strictly before outer test season T",
        "evaluation_rows": "exactly the frozen headline and A1.4 strict-sensitivity rows",
        "estimator": "Ridge on the residual over the structural FEATURE_COLS",
        "imputation": "median, fitted on each outer training window only",
        "scaling": "standardization, fitted on each outer training window only",
        "alpha_selection": "inner expanding-season validation inside the outer training window; "
                           "smallest alpha among ties",
        "alpha_grid": [0.01, 0.1, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0, 1000.0],
        "fallback_alpha": 10.0,
        "fallback_condition": "fewer than two usable inner validation seasons in the outer fold",
        "random_cross_validation": False,
        "clipping": "none",
        "clipping_rationale": "no clipping rule is pre-declared for M1-M4, so M5-only clipping "
                              "would make the tracks incomparable",
        "training_by_fold": m5_training_by_fold,
        "first_fold_limitation": "the earliest headline fold (2015) has exactly one line-covered "
                                 "training season (2014, 32 rows) and zero usable inner validation "
                                 "seasons, so it necessarily takes the fallback alpha",
        "reporting_requirement": "every report must show M5 training seasons and row count by fold",
        "predict_season_status": ("no archived consensus line exists for the predict season, so M5 "
                                  "fails closed there while M1-M4 remain possible"),
        "permitted_claim": "whether structural information improved MAE relative to the archived "
                           "consensus out of sample",
    },
    "preseason_feature_feasibility": {
        "artifact": _rel(FEAS_PATH), "counts": feasibility_doc["counts"],
        "families_added_to_features": 0,
        "note": "CONDITIONAL and UNAVAILABLE families are excluded by A2.4; the two AVAILABLE "
                "families are not added by Amendment 2 either",
    },
    "features": {"order_is_contract": True, "columns": FEATURE_COLS, "n": len(FEATURE_COLS),
                 "available_families": AVAILABLE_FAMILIES,
                 "deferred_families": DEFERRED_FAMILIES,
                 "pbp_family_columns": PBP_FEATURE_COLS,
                 "excluded_families": UNAVAILABLE_FAMILIES,
                 "pythagorean_exponent": PYTHAG_EXP,
                 "coach_prior_min_games": 16},
    "eligibility": {
        "rule": "train = every settled team-season with season < T (line-covered or not); "
                "eval = line-covered rows in T. Market rows define evaluation and B0 only.",
        "headline": eligibility, "strict_sensitivity": eligibility_strict,
        "note": "every headline number must also be reported on the A1.4 strict subset"},
    "folds": {"headline": FOLDS, "strict": FOLDS_STRICT,
              "source": "read verbatim from data_audit.json; never recomputed here"},
    "market": {"rows_covered": int(panel_out["line_covered"].sum()),
               "seasons_covered": sorted(int(s) for s in panel.loc[panel["line_covered"], "season"].unique()),
               "strictly_dated_rows": int(panel_out["market_strictly_dated"].sum()),
               "role": "defines evaluation rows and baseline B0 only — never training eligibility",
               "naming": LOCK["market_naming"]},
    "lock": LOCK,
    "leakage_tests": {"blinding_probe": blind_report,
                      "red_control_detected_injected_leak": True,
                      "target_derived_columns_excluded": sorted(TARGET_DERIVED)},
    "inputs": {"audit": _rel(AUDIT), "audit_verdict": VERDICT,
               "schedule_snapshot": _rel(SNAP), "schedule_sha256": SCHED_HASH,
               "lines_file": _rel(LINES), "lines_sha256": LINES_HASH,
               "outcome_sha256": OUTCOME_HASH,
               "pbp_snapshot": _rel(PBP_SNAP), "pbp_sha256": PBP_HASH,
               "pbp_source": PBP_SOURCE, "pbp_provenance": _rel(PBP_PROV)},
    "outputs": {"panel": _rel(PANEL), "panel_frame_sha256": PANEL_HASH,
                "venue_context": _rel(VENUE_PATH), "venue_frame_sha256": VENUE_HASH,
                "venue_rows": int(len(venue_out)),
                "feasibility": _rel(FEAS_PATH)},
    "provenance": PROVENANCE,
}
# Recursive guard over the WHOLE metadata, minus the lock block that exists to name what it
# forbids. This is the check the flat guard could not perform.
guard({k: v for k, v in metadata.items() if k != "lock"}, "metadata")
if WRITE_ARTIFACTS:
    META.write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")

print(f"panel    : {_rel(PANEL)}  {len(panel_out):,} × {len(PANEL_COLS)}  frame {PANEL_HASH[:16]}…")
print(f"venue    : {_rel(VENUE_PATH)}  {len(venue_out):,} × {len(VENUE_COLS)}  frame {VENUE_HASH[:16]}…")
print(f"M5 folds : first {m5_training_by_fold[0]['test_season']} trains on "
      f"{m5_training_by_fold[0]['train_seasons']} ({m5_training_by_fold[0]['n_train']} rows, "
      f"fallback alpha={m5_training_by_fold[0]['takes_fallback_alpha']})")
print(f"metadata : {_rel(META)}")
print(f"tier     : {TIER}  gate C open: {TIER_C_OPEN}")
print(f"folds    : headline {FOLDS}")
print(f"           strict   {FOLDS_STRICT}")

### Interpreting the output

The panel is the single input `02` reads; the metadata is its contract — feature order, eligibility
tables, hashes and the lock. `tier_c_open` appears in both, so a downstream notebook cannot claim it
did not know.

### What these tests guard

That the written file round-trips identically, the recorded hashes match disk, the metadata's fold
lists equal the audit's verbatim, and the lock is present and closed. Also that the metadata itself
carries no Tier-C vocabulary outside the lock block that is allowed to name what it forbids.

In [ ]:
if RUN_TESTS and WRITE_ARTIFACTS:
    _back = pd.read_parquet(PANEL)
    assert list(_back.columns) == PANEL_COLS and len(_back) == len(panel_out)
    assert sha256_frame(_back) == PANEL_HASH, "panel does not round-trip"
    _m = json.loads(META.read_text(encoding="utf-8"))
    assert _m["folds"]["headline"] == audit["folds"]["test_seasons"], "fold list drifted from the audit"
    assert _m["folds"]["strict"] == audit["folds_strict_sensitivity"]["test_seasons"]
    assert _m["lock"]["tier_c_open"] is False and _m["panel"]["frame_sha256"] == PANEL_HASH
    assert _m["inputs"]["schedule_sha256"] == AUDIT_SCHED_HASH == SCHED_HASH
    assert _m["inputs"]["lines_sha256"] == AUDIT_LINES_HASH == LINES_HASH
    assert _m["eligibility"]["headline"][0]["n_train"] == 32 * 13
    # Amendment 2 blocks present and internally consistent
    assert _m["venue_contract"]["artifact"].endswith("season_schedule_context.parquet")
    assert _m["m5_contract"]["fallback_alpha"] == 10.0 and _m["m5_contract"]["clipping"] == "none"
    assert _m["m5_contract"]["training_by_fold"][0]["takes_fallback_alpha"] is True, \
        "the 2015 M5 fold must be recorded as taking the fallback alpha"
    assert _m["model_tracks"]["m1_m4_independent_structural"]["market_line_in_features"] is False
    assert "market_line" not in _m["features"]["columns"], "market_line leaked into the structural features"
    assert _m["preseason_feature_feasibility"]["families_added_to_features"] == 0
    assert sha256_frame(pd.read_parquet(VENUE_PATH)) == VENUE_HASH, "venue context does not round-trip"
    # Tier-C vocabulary may appear ONLY inside the lock block, which exists to name what it forbids
    _scan = {k: v for k, v in _m.items() if k != "lock"}
    guard(_scan, "metadata-readback")      # recursive: keys AND values, at any depth
    assert _m["inputs"]["pbp_sha256"] == PBP_HASH, "PBP snapshot hash not recorded"
    assert _m["features"]["pbp_family_columns"] == PBP_FEATURE_COLS, "PBP family not recorded"
    assert not _m["features"]["deferred_families"], "a family is still deferred"
    print(f"✓ Section 13 tests passed | panel round-trips ({PANEL_HASH[:12]}…), folds match the audit "
          f"verbatim, venue context pinned ({VENUE_HASH[:12]}…), lock closed, metadata clean")

### Reading the test result

The artifacts on disk match memory, the audit, and the lock. Does **not** prove `02` will honour any
of it — it makes a violation detectable rather than possible-and-silent.

## Section 14 — Sanity check: look at the data

The last thing before handing the panel to `02`: read the first five rows with your own eyes.
Shown for a **training** season (target present, market line present) and for the **predict**
season (features present, target and line blank by construction).

This reads the frame that was written in Section 11 — it re-derives nothing, so what is displayed
is what `02` will load.

In [ ]:
SANITY_COLS = ["season", "franchise",
               "designated_home_games", "true_home_venue_games", "neutral_site_games",
               "international_games",
               "prior_wins", "prior_win_pct", "prior_point_diff",
               "prior_off_epa_play", "prior_def_epa_play", "prior_off_success_rate",
               "form_3yr_win_pct", "sos_prior_win_pct", "coach_changed", "coach_prior_win_pct",
               TARGET_COL, "market_line", "line_covered"]

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

print(f"panel: {len(panel_out):,} rows x {len(panel_out.columns)} columns "
      f"({len(FEATURE_COLS)} features + target + market + flags)")
print()
print("FEATURES (pinned order):")
for _i, _c in enumerate(FEATURE_COLS, 1):
    print(f"  {_i:2d}. {_c}")
print()
print(f"TARGET : {TARGET_COL}")
print("MARKET : market_line  (baseline B0 — never an input)")
print("FLAGS  : line_covered, has_target, is_predict_season, market_strictly_dated")
print()

_train_head = panel_out[panel_out["line_covered"]].head(5)
_pred_head = panel_out[panel_out["is_predict_season"]].head(5)

print(f"--- first 5 evaluation rows ({int(_train_head['season'].iloc[0])}: target + market line present) ---")
print(_train_head[SANITY_COLS].to_string(index=False))
print()
print(f"--- first 5 predict-season rows ({PREDICT_SEASON}: features only, no target, no line) ---")
print(_pred_head[SANITY_COLS].to_string(index=False))
print()
print(f"--- target summary over the {int(panel_out['has_target'].sum())} settled rows ---")
print(panel_out.loc[panel_out["has_target"], TARGET_COL].describe().to_frame().T.to_string())

panel_out.head(5)          # rendered as a table by the notebook UI

### Interpreting the output

The evaluation rows show the shape the model actually sees: `prior_*` values from *S−1* (a 2014 row
carries 2013's record and EPA), `wins_half_ties` as the target, and `market_line` alongside as B0 —
never as an input.

The predict-season block is the useful contrast: identical features, **blank target and blank
market line**. That is the leakage rule visible in the data rather than in prose.

The target summary should read like NFL seasons — mean near half the games, range 0 to about 17.

### What these tests guard

That the display is the written panel and not a re-derivation, that the two blocks really are what
they claim (targets present in one, absent in the other), and that the target distribution is
sane — a mean far from half a season would mean the join or the target is wrong.

In [ ]:
if RUN_TESTS:
    assert len(_train_head) == 5 and len(_pred_head) == 5
    assert _train_head[TARGET_COL].notna().all() and _train_head["market_line"].notna().all()
    assert _pred_head[TARGET_COL].isna().all() and _pred_head["market_line"].isna().all(),         "the predict season must carry no target and no market line"
    assert set(SANITY_COLS) <= set(panel_out.columns), "sanity view references a column not in the panel"
    assert sha256_frame(panel_out) == PANEL_HASH, "the displayed frame is not the one written"
    _t = panel_out.loc[panel_out["has_target"], TARGET_COL]
    assert 7.5 < _t.mean() < 9.0 and _t.min() >= 0 and _t.max() <= 17,         f"target distribution looks wrong: mean {_t.mean():.2f}, range {_t.min()}–{_t.max()}"
    guard(panel_out[SANITY_COLS], "sanity-view")
    print(f"✓ Section 14 tests passed | displayed frame is the written panel ({PANEL_HASH[:12]}…), "
          f"target mean {_t.mean():.2f} over {len(_t)} settled rows, predict season blank as required")

### Reading the test result

The rows on screen are the written artifact, the predict season is blank where it must be, and the
target averages about half a season. Does **not** validate any feature's *value* — only that the
panel is shaped and populated the way the rest of the pipeline assumes.

## Conclusion and next steps

**Built:** `data/team_season_panel.parquet` — 800 rows (25 seasons × 32), **24 features**, target
`wins_half_ties` on 768 rows, market line on 352 — plus `data/season_schedule_context.parquet`
(the A2.1 venue authority), `artifacts/dataset_metadata.json`,
`artifacts/preseason_feature_feasibility.json`, and the pinned `data/pbp_team_season_epa.parquet`.

**Amendment 2 applied:** `home_games` retired for the four venue counts; the 2026
Jacksonville–Philadelphia game at Tottenham is caught despite being labelled `location = "Home"`;
M5's contract is frozen in metadata with its first-fold fallback recorded; 13 preseason feature
families are verdicted and **none added**.

**The three rules, verified not asserted:** training uses every settled season before *T* (2015 →
2002–2014, 416 rows) and includes 2023 for the 2024/2025 folds; the market defines evaluation and B0
only; prices are withheld from the panel and a recursively red/green-tested guard blocks Tier-C vocabulary.

**Leakage:** blinding season *T*'s results **and** its EPA changes no season-*T* feature, on all 10
folds and the predict season — and the probe was proven able to catch an injected leak from each
source.

**Nothing deferred.** All seven AVAILABLE families are built, including `prior_season_pbp_epa` from
a pinned team-season aggregate. No feature decision remains open once `02` produces numbers, so no
post-hoc feature amendment is needed.

**Next:** `02_model_comparison.ipynb` — M1–M3 against B0 (archived consensus), B1–B3, on the frozen
folds, every number reported for the headline **and** the A1.4 strict subset, with `assert_no_tier_c`
called before anything is written.